# Is the 5-minute window creating the patterns?

Mouse 720762, PL5 LFP. The tracker uses a 5-min sliding baseline (`baseline_window = 300`) and 5-min detection
windows (`window = 300`); the pole analysis uses 5-min analysis windows. None of these were tuned. Five tests:

- **E1. Baseline-window sweep.** Do the state associations (Q1a of tracker_vs_static) hold for baselines of
  1, 2, 5, 10 and 20 min?
- **E2. Block-shuffle surrogate.** Within each condition segment, 60-s blocks of the band envelopes are shuffled.
  This keeps the fast content (pole switching, its variance, the condition) and destroys the slow order. If a
  sliding window turned fast switching into fake slow movement, the surrogates would show the same baseline
  movement and the same associations as the real data.
- **E3. Pole-analysis window sweep.** Do the pole mappings (Q1f) hold for 2-, 5- and 10-min analysis windows,
  and do they beat the block-shuffle surrogate?
- **E4. Re-tracking.** The whole tracker re-run with baselines of 2, 10 and 20 min and detection windows of 3
  and 10 min. Are the same motifs found, with the same state map?
- **E5. Synthetic ground truth.** A planted motif switching between its poles, with and without a planted slow
  drift. How much fake baseline movement does each window create, how much of a real drift does it recover,
  and how much leaks into the expression?

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import motif_tracking as mt, spike_validation as sv, session_data as sd
import warnings; warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

NWB = REPO / "data" / "raw" / "720762-20240221-v001.nwb"
OUT = REPO / "data" / "out" / "2026-11-10_window_tests"; OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")

res = mt.load(REPO / "data" / "processed" / "motif_tracking" / "720762" / "lfp_PL5.pkl")
sig = mt.prepare(NWB, "lfp_PL5", ("lfp", "probeF", "PL5"), res.config)
info, (run_t, run), (pupil_t, pupil) = mt.behaviour(NWB)
t, Zg = sig.t, sig.Z
K = len(res.library)
U = np.array([m.template for m in res.library])
R = json.load(open(REPO / "data" / "out" / "2026-11-01_mouse720762" / "results.json"))
runs = {(r["window"], r["version"]): np.array(r["motifs"], float)[:, :r["n_motifs"]] for r in R["motif_runs"] if r["signal"] == "lfp_PL5"}
ORIG = runs["awake", "strict"][:, 0] / np.linalg.norm(runs["awake", "strict"][:, 0])

csum = np.r_[0, np.cumsum(run)]
lo, hi = np.searchsorted(run_t, t - 0.5), np.searchsorted(run_t, t + 0.5)
speed = (csum[hi] - csum[lo]) / np.maximum(hi - lo, 1)
okp = np.isfinite(pupil)
pup = pd.Series(np.interp(t, pupil_t[okp], pupil[okp])).rolling(5, center=True, min_periods=1).mean().to_numpy()
iso = np.interp(t, info.iso_t, info.iso_level)
state = sd.state_of_time(t, info)
BWS = [60, 120, 300, 600, 1200]
BWLAB = ["1 min", "2 min", "5 min", "10 min", "20 min"]

def ranks(A):
    A = np.asarray(A, float); A = A if A.ndim == 2 else A[:, None]
    r = np.argsort(np.argsort(A, axis=0), axis=0).astype(float)
    return (r - r.mean(0)) / (r.std(0) + 1e-12)

def best_r(F, y):
    RF, ry = ranks(F), ranks(y)[:, 0]
    return float(np.abs(RF.T @ ry / ry.size).max())

def best_feature(F, y, rng, n_shift=200, min_shift=300):
    RF, ry = ranks(F), ranks(y)[:, 0]
    rs = np.abs(RF.T @ ry / ry.size)
    ks = rng.integers(min_shift, ry.size - min_shift, n_shift)
    null = np.array([np.abs(RF.T @ np.roll(ry, k) / ry.size).max() for k in ks])
    return float(rs.max()), float(np.percentile(null, 95)), float((1 + (null >= rs.max()).sum()) / (n_shift + 1))

def block_shuffle(Z, segment, rng, block=60):
    """Shuffle 60-s blocks of rows within each segment (segments are cut at condition boundaries)."""
    out = Z.copy()
    for s in np.unique(segment):
        i = np.flatnonzero(segment == s)
        nb = i.size // block
        if nb < 2:
            continue
        blocks = i[:nb * block].reshape(nb, block)
        out[blocks.ravel()] = Z[blocks[rng.permutation(nb)].ravel()]
    return out

TARGETS = {"Isoflurane level (whole session)": (iso, np.ones(t.size, bool)),
           "Isoflurane level (within maintenance)": (iso, state == "isoflurane"),
           "Pupil (within awake)": (pup, state == "awake"),
           "Pupil (within maintenance)": (pup, state == "isoflurane"),
           "Running speed (within awake)": (speed, state == "awake")}
seg = sig.segment
SEG_DOSE = max(np.unique(seg[state == "isoflurane"]), key=lambda s_: np.ptp(iso[seg == s_]))   # the isoflurane stimulation block
def demean(v, m):
    """Remove each condition segment's mean (rows in m only)."""
    v = np.asarray(v, float).copy(); out = v[m]; sg = seg[m]
    for s_ in np.unique(sg):
        k = sg == s_; out[k] = out[k] - out[k].mean(0)
    return out
WITHIN = {"Dose, inside the isoflurane\nstimulation block only": ("iso", seg == SEG_DOSE),
          "Pupil, within each\nmaintenance segment": ("pup", state == "isoflurane"),
          "Pupil, within each\nawake segment": ("pup", state == "awake"),
          "Running, within each\nawake segment": ("speed", state == "awake")}
COV = {"iso": iso, "pup": pup, "speed": speed}
print(K, "library motifs; dose-varying segment:", SEG_DOSE, t[seg == SEG_DOSE][[0, -1]] / 60)

9 library motifs; dose-varying segment: 7 [ 77.47558142 105.30886982]


## E1. Do the state associations depend on the baseline window?

For each baseline window, the baseline is rebuilt and projected on the (fixed) library directions, as the tracker
does. Score as in Q1a: best single feature's |Spearman r| with the target, null = the same maximum after shifting
the target by at least 5 min. The raw 10 bands (no baseline) are the reference.

In [2]:
rng = np.random.default_rng(0)
BASES = {bw: mt.split_baseline(Zg, sig.segment, bw)[0] for bw in BWS}
rows = []
for tname, (y, m0) in TARGETS.items():
    for bw, lab in zip(BWS, BWLAB):
        B = BASES[bw]
        m = m0 & np.isfinite(B).all(1)
        for fname, F in (("Tracker baseline (9 motifs)", B @ U.T), ("10 bands, sliding baseline", B)):
            r, n95, p = best_feature(F[m], y[m], rng)
            rows.append({"target": tname, "window": lab, "features": fname, "|r|": r, "null95": n95, "p": p})
    m = m0 & np.isfinite(BASES[1200]).all(1)
    r, n95, p = best_feature(Zg[m], y[m], rng)
    rows.append({"target": tname, "window": "none", "features": "10 bands, no baseline", "|r|": r, "null95": n95, "p": p})
e1 = pd.DataFrame(rows); e1.to_csv(OUT / "E1_state_association_by_window.csv", index=False)
e1.pivot_table(index=["target", "features"], columns="window", values="|r|", sort=False).reindex(columns=BWLAB + ["none"]).round(2)

window                                                             1 min  \
target                                features                             
Isoflurane level (whole session)      Tracker baseline (9 motifs)   0.78   
                                      10 bands, sliding baseline    0.78   
                                      10 bands, no baseline          NaN   
Isoflurane level (within maintenance) Tracker baseline (9 motifs)   0.70   
                                      10 bands, sliding baseline    0.61   
                                      10 bands, no baseline          NaN   
Pupil (within awake)                  Tracker baseline (9 motifs)   0.51   
                                      10 bands, sliding baseline    0.43   
                                      10 bands, no baseline          NaN   
Pupil (within maintenance)            Tracker baseline (9 motifs)   0.65   
                                      10 bands, sliding baseline    0.68   
                                      10 bands, no baseline          NaN   
Running speed (within awake)          Tracker baseline (9 motifs)   0.30   
                                      10 bands, sliding baseline    0.31   
                                      10 bands, no baseline          NaN   

window                                                             2 min  \
target                                features                             
Isoflurane level (whole session)      Tracker baseline (9 motifs)   0.80   
                                      10 bands, sliding baseline    0.79   
                                      10 bands, no baseline          NaN   
Isoflurane level (within maintenance) Tracker baseline (9 motifs)   0.71   
                                      10 bands, sliding baseline    0.62   
                                      10 bands, no baseline          NaN   
Pupil (within awake)                  Tracker baseline (9 motifs)   0.53   
                                      10 bands, sliding baseline    0.47   
                                      10 bands, no baseline          NaN   
Pupil (within maintenance)            Tracker baseline (9 motifs)   0.67   
                                      10 bands, sliding baseline    0.69   
                                      10 bands, no baseline          NaN   
Running speed (within awake)          Tracker baseline (9 motifs)   0.28   
                                      10 bands, sliding baseline    0.30   
                                      10 bands, no baseline          NaN   

window                                                             5 min  \
target                                features                             
Isoflurane level (whole session)      Tracker baseline (9 motifs)   0.80   
                                      10 bands, sliding baseline    0.79   
                                      10 bands, no baseline          NaN   
Isoflurane level (within maintenance) Tracker baseline (9 motifs)   0.72   
                                      10 bands, sliding baseline    0.65   
                                      10 bands, no baseline          NaN   
Pupil (within awake)                  Tracker baseline (9 motifs)   0.59   
                                      10 bands, sliding baseline    0.51   
                                      10 bands, no baseline          NaN   
Pupil (within maintenance)            Tracker baseline (9 motifs)   0.67   
                                      10 bands, sliding baseline    0.69   
                                      10 bands, no baseline          NaN   
Running speed (within awake)          Tracker baseline (9 motifs)   0.26   
                                      10 bands, sliding baseline    0.28   
                                      10 bands, no baseline          NaN   

window                                                             10 min  \
target                                featur

In [3]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.9), sharey=True)
COL = {"Tracker baseline (9 motifs)": "#C2504A", "10 bands, sliding baseline": "#8E5EA2"}
for ax, tname in zip(axes, TARGETS):
    for f, c in COL.items():
        sub = e1[(e1.target == tname) & (e1.features == f)].set_index("window").reindex(BWLAB)
        x = np.arange(len(BWLAB))
        ax.plot(x, sub["|r|"], color=c, lw=1.6, label=f)
        ax.plot(x, sub["null95"], color=c, lw=1, ls=":")
        sg = sub.p <= 0.05
        ax.scatter(x[sg], sub["|r|"][sg], color=c, s=40, zorder=3)
        ax.scatter(x[~sg], sub["|r|"][~sg], color="white", edgecolor=c, s=40, zorder=3)
    raw = e1[(e1.target == tname) & (e1.window == "none")].iloc[0]
    ax.axhline(raw["|r|"], color="#999999", lw=1.2, ls="--", label="10 bands, no baseline")
    ax.axvline(2, color="0.88", lw=8, zorder=0)
    ax.set_xticks(range(len(BWLAB)), BWLAB, fontsize=8); ax.set_title(tname.replace(" (", "\n("), fontsize=9.5)
    ax.set_xlabel("sliding-baseline window")
axes[0].set_ylabel("best single feature |Spearman r|"); axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=7.5, frameon=False, loc="lower left")
fig.suptitle("E1. State associations vs baseline window (filled: beats the time-shift null, dotted line = null 95th percentile; "
             "shaded: the 5 min we used)", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "W1_state_association_vs_window"); plt.show()

## E2. Is the baseline movement real, or made by the window from fast switching?

20 block-shuffle surrogates (60-s blocks shuffled within each condition segment). For each window:
- **baseline movement**: SD over time of each motif's baseline position, within awake and within maintenance;
- **association**: the best |r| between the tracker baseline and each target.

Real values far above the surrogates mean the slow movement is in the data. Real values inside the surrogate
range mean the window could have produced it from fast fluctuations alone.

Because segments are cut at condition changes, the shuffle keeps each condition's mean. Everything is therefore
scored twice: pooled over the state (condition differences count), and **within segments** (each segment's mean
removed, or, for dose, only inside the one segment where dose changes: the isoflurane stimulation block). Only
the within-segment version tests slow tracking inside a condition.

In [4]:
N_SUR = 20
rng = np.random.default_rng(1)
SURS = [block_shuffle(Zg, sig.segment, rng) for _ in range(N_SUR)]
mv_rows, as_rows = [], []
for bw, lab in zip(BWS, BWLAB):
    sur_bases = [mt.split_baseline(S, sig.segment, bw)[0] for S in SURS]
    for kind, B_list in (("real", [BASES[bw]]), ("surrogate", sur_bases)):
        for k, B in enumerate(B_list):
            P = B @ U.T
            for st in ("awake", "isoflurane"):
                m = (state == st) & np.isfinite(P).all(1)
                Pw = demean(P, m)
                for i in range(3):
                    mv_rows.append({"window": lab, "kind": kind, "rep": k, "state": st, "motif": f"M{i + 1}", "baseline SD": float(P[m, i].std()),
                                    "within-segment SD": float(Pw[:, i].std())})
            for tname, (y, m0) in TARGETS.items():
                m = m0 & np.isfinite(P).all(1)
                as_rows.append({"window": lab, "kind": kind, "rep": k, "scope": "pooled", "target": tname, "|r|": best_r(P[m], y[m])})
            for tname, (cv, m0) in WITHIN.items():
                m = m0 & np.isfinite(P).all(1)
                as_rows.append({"window": lab, "kind": kind, "rep": k, "scope": "within", "target": tname,
                                "|r|": best_r(demean(P, m), demean(COV[cv], m))})
mv = pd.DataFrame(mv_rows); asc = pd.DataFrame(as_rows)
mv.to_csv(OUT / "E2_baseline_movement_real_vs_shuffle.csv", index=False); asc.to_csv(OUT / "E2_association_real_vs_shuffle.csv", index=False)
summ = mv.groupby(["state", "motif", "window", "kind"])[["baseline SD", "within-segment SD"]].mean().unstack("kind")
summ

baseline SD           within-segment SD          
kind                           real surrogate              real surrogate
state      motif window                                                  
awake      M1    1 min     0.319209  0.296358          0.220764  0.186420
                 10 min    0.268915  0.242826          0.148934  0.079901
                 2 min     0.298145  0.268436          0.193567  0.138670
                 20 min    0.258334  0.239208          0.136598  0.073181
                 5 min     0.282941  0.252612          0.163850  0.101724
           M2    1 min     0.185752  0.175653          0.124716  0.109172
                 10 min    0.145050  0.144953          0.065113  0.045132
                 2 min     0.171582  0.160661          0.102969  0.082082
                 20 min    0.142730  0.143114          0.060989  0.038912
                 5 min     0.154225  0.150025          0.075831  0.057917
           M3    1 min     0.206464  0.184748          0.169561  0.142374
                 10 min    0.161995  0.136723          0.121569  0.062898
                 2 min     0.191091  0.161283          0.151530  0.107615
                 20 min    0.155872  0.135273          0.113477  0.059711
                 5 min     0.173758  0.145731          0.129913  0.079888
isoflurane M1    1 min     2.464480  2.306199          1.735892  1.497596
                 10 min    2.149705  1.910443          1.395541  0.791771
                 2 min     2.385080  2.190326          1.618296  1.312094
                 20 min    1.861555  1.834501          1.001479  0.623693
                 5 min     2.313360  2.026224          1.596749  1.029458
           M2    1 min     0.826881  0.756165          0.710970  0.631764
                 10 min    0.668221  0.552937          0.500596  0.355648
                 2 min     0.791353  0.714118          0.647868  0.570663
                 20 min    0.559349  0.507576          0.356284  0.281882
                 5 min     0.749441  0.627027          0.603014  0.459809
           M3    1 min     0.918621  0.845177          0.746002  0.654395
                 10 min    0.779599  0.640601          0.602001  0.362752
                 2 min     0.879082  0.791204          0.688812  0.580123
                 20 min    0.654282  0.591344          0.460925  0.275455
                 5 min     0.867532  0.712975          0.700306  0.475178

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
MC = {"M1": "#1D232B", "M2": "#E0A030", "M3": "#2F7F8F"}
x = np.arange(len(BWLAB))
for ax, st in zip(axes, ("awake", "isoflurane")):
    for mo, c in MC.items():
        real = [mv[(mv.state == st) & (mv.motif == mo) & (mv.window == w) & (mv.kind == "real")]["within-segment SD"].iloc[0] for w in BWLAB]
        sur = [mv[(mv.state == st) & (mv.motif == mo) & (mv.window == w) & (mv.kind == "surrogate")]["within-segment SD"].to_numpy() for w in BWLAB]
        ax.plot(x, real, color=c, lw=1.8, marker="o", label=f"{mo} real")
        ax.fill_between(x, [np.percentile(s, 5) for s in sur], [np.percentile(s, 95) for s in sur], color=c, alpha=0.18, lw=0)
        ax.plot(x, [np.median(s) for s in sur], color=c, lw=1, ls="--")
    ax.axvline(2, color="0.88", lw=8, zorder=0)
    ax.set_xticks(x, BWLAB); ax.set_xlabel("sliding-baseline window"); ax.set_title(st, loc="left", fontsize=10)
axes[0].set_ylabel("how much the baseline moves inside a condition\n(SD of baseline position, segment means removed)")
axes[0].legend(fontsize=8, frameon=False)
fig.suptitle("E2. Real baseline movement (solid) vs 60-s block-shuffled data (dashed: median, band: 5–95 %); "
             "the shuffle keeps fast switching, destroys slow order", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "W2_baseline_movement_vs_shuffle"); plt.show()

In [6]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7.6), sharey=True)
for row, (scope, tg) in enumerate((("pooled", list(TARGETS)), ("within", list(WITHIN)))):
    for ax, tname in zip(axes[row], tg):
        sub = asc[(asc.scope == scope) & (asc.target == tname)]
        real = [sub[(sub.window == w) & (sub.kind == "real")]["|r|"].iloc[0] for w in BWLAB]
        sur = [sub[(sub.window == w) & (sub.kind == "surrogate")]["|r|"].to_numpy() for w in BWLAB]
        beat = np.array([r_ > np.percentile(s_, 95) for r_, s_ in zip(real, sur)])
        ax.fill_between(x, [np.percentile(s_, 5) for s_ in sur], [np.percentile(s_, 95) for s_ in sur], color="#C2504A", alpha=0.15, lw=0, label="shuffled, 5–95 %")
        ax.plot(x, [np.median(s_) for s_ in sur], color="#C2504A", lw=1, ls="--", label="shuffled, median")
        ax.plot(x, real, color="#C2504A", lw=1.8, label="real")
        ax.scatter(x[beat], np.array(real)[beat], color="#C2504A", s=40, zorder=3, label="real, above shuffle 95 %")
        ax.scatter(x[~beat], np.array(real)[~beat], color="white", edgecolor="#C2504A", s=40, zorder=3)
        ax.axvline(2, color="0.88", lw=8, zorder=0)
        ax.set_xticks(x, BWLAB, fontsize=8); ax.set_title(tname.replace(" (", "\n("), fontsize=9.5)
        if row == 1: ax.set_xlabel("sliding-baseline window")
    axes[row, 0].set_ylabel(("POOLED over the state\n" if row == 0 else "WITHIN condition segments\n") + "tracker baseline, best |r|")
axes[1, 4].axis("off")
axes[0, 0].set_ylim(0, 1); axes[0, 0].legend(fontsize=7.5, frameon=False, loc="lower left")
fig.suptitle("E2. Baseline–state associations, real vs 60-s block-shuffled within condition segments. Top: pooled (differences between "
             "conditions count). Bottom: inside conditions only", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "W3_association_vs_shuffle"); plt.show()

## E3. Do the pole mappings depend on the analysis window?

Pole measures as in Q1f (median split along the axis, session-wide frame, all seconds), with analysis windows
of 2, 5 and 10 min (60-s steps). The key mappings from Q1f are re-scored at each length, against two nulls:
the window-series shift (as before) and 20 block-shuffle surrogates.

In [7]:
AXES = {"M1 (broadband)": U[0], "M2": U[1], "M3": U[2], "Original (awake)": ORIG}
MAPS = [("separation", "iso", "all", "Separation vs dose\n(whole session)"),
        ("mid", "iso", "isoflurane", "Midpoint vs dose\n(within maintenance)"),
        ("minus", "pupil", "isoflurane", "− pole vs pupil\n(within maintenance)"),
        ("plus", "iso", "isoflurane", "+ pole vs dose\n(within maintenance)"),
        ("mid", "iso", "dose_seg", "Midpoint vs dose\n(inside the stimulation block only)")]
def win_measures(Z, u, width, step=60):
    x = Z @ u
    starts = np.arange(t[0], t[-1] - width + 1, step)
    rows = []
    for a in starts:
        sel = (t >= a) & (t < a + width)
        v = x[sel]; med = np.median(v)
        p_, n_ = v[v > med].mean(), v[v <= med].mean()
        rows.append({"centre": a + width / 2, "plus": p_, "minus": n_, "mid": (p_ + n_) / 2, "separation": p_ - n_,
                     "iso": iso[sel].mean(), "pupil": pup[sel].mean()})
    W = pd.DataFrame(rows); W["state"] = sd.state_of_time(W.centre.to_numpy(), info)
    ts = t[seg == SEG_DOSE]
    W.loc[(W.centre - width / 2 >= ts[0]) & (W.centre + width / 2 <= ts[-1]), "state"] = "dose_seg"
    return W
def score_map(W, meas, cov, st):
    if st == "all":
        m = np.ones(len(W), bool)
    elif st == "isoflurane":
        m = W.state.isin(["isoflurane", "dose_seg"]).to_numpy()
    else:
        m = (W.state == st).to_numpy()
    return spearmanr(W[meas][m], W[cov][m]).correlation, m
rng = np.random.default_rng(2)
SUR_POLE = SURS[:20]
rows = []
for width in (120, 300, 600):
    for an, u in AXES.items():
        W = win_measures(Zg, u, width)
        Ws = [win_measures(S, u, width) for S in SUR_POLE]
        for meas, cov, st, lab in MAPS:
            r, m = score_map(W, meas, cov, st)
            a, b = W[meas].to_numpy()[m], W[cov].to_numpy()[m]
            ks = rng.integers(min(10, a.size // 3), a.size - min(10, a.size // 3), 300)
            shift_null = np.array([spearmanr(np.roll(a, k), b).correlation for k in ks])
            sur = np.array([score_map(Wx, meas, cov, st)[0] for Wx in Ws])
            rows.append({"window": f"{width // 60} min", "axis": an, "mapping": lab, "r": r,
                         "p shift": float((1 + (np.abs(shift_null) >= abs(r)).sum()) / 301),
                         "shuffle median": float(np.median(sur)), "shuffle 5%": float(np.percentile(sur, 5)),
                         "shuffle 95%": float(np.percentile(sur, 95)),
                         "beats shuffle": bool(abs(r) > np.percentile(np.abs(sur), 95))})
e3 = pd.DataFrame(rows); e3.to_csv(OUT / "E3_pole_mappings_by_window.csv", index=False)
e3.pivot_table(index=["mapping", "axis"], columns="window", values="r", sort=False).round(2)

window                                                               2 min  \
mapping                                            axis                      
Separation vs dose\n(whole session)                M1 (broadband)     0.67   
                                                   M2                 0.53   
                                                   M3                 0.64   
                                                   Original (awake)   0.64   
Midpoint vs dose\n(within maintenance)             M1 (broadband)    -0.62   
                                                   M2                -0.37   
                                                   M3                -0.72   
                                                   Original (awake)  -0.70   
− pole vs pupil\n(within maintenance)              M1 (broadband)    -0.62   
                                                   M2                -0.39   
                                                   M3                -0.60   
                                                   Original (awake)  -0.63   
+ pole vs dose\n(within maintenance)               M1 (broadband)    -0.61   
                                                   M2                -0.33   
                                                   M3                -0.69   
                                                   Original (awake)  -0.66   
Midpoint vs dose\n(inside the stimulation block... M1 (broadband)    -0.48   
                                                   M2                -0.40   
                                                   M3                -0.61   
                                                   Original (awake)  -0.57   

window                                                               5 min  \
mapping                                            axis                      
Separation vs dose\n(whole session)                M1 (broadband)     0.67   
                                                   M2                 0.53   
                                                   M3                 0.65   
                                                   Original (awake)   0.63   
Midpoint vs dose\n(within maintenance)             M1 (broadband)    -0.70   
                                                   M2                -0.45   
                                                   M3                -0.77   
                                                   Original (awake)  -0.74   
− pole vs pupil\n(within maintenance)              M1 (broadband)    -0.72   
                                                   M2                -0.44   
                                                   M3                -0.69   
                                                   Original (awake)  -0.70   
+ pole vs dose\n(within maintenance)               M1 (broadband)    -0.68   
                                                   M2                -0.40   
                                                   M3                -0.74   
                                                   Original (awake)  -0.68   
Midpoint vs dose\n(inside the stimulation block... M1 (broadband)    -0.58   
                                                   M2                -0.48   
                                                   M3                -0.65   
                                                   Original (awake)  -0.62   

window                                                               10 min  
mapping                                            axis                      
Separation vs dose\n(whole session)                M1 (broadband)      0.65  
                                                   M2                  0.62  
                                                   M3                  0.67  
                                                   Original (awake)    0.65  
Midpoint vs dose\n(within maintenance)             M1 (broadband)     -0.81  
                    

In [8]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4.2), sharey=True)
AC = {"M1 (broadband)": "#1D232B", "M2": "#E0A030", "M3": "#2F7F8F", "Original (awake)": "#9AA5B1"}
WL = ["2 min", "5 min", "10 min"]
for ax, (_, _, _, lab) in zip(axes, MAPS):
    for k, (an, c) in enumerate(AC.items()):
        sub = e3[(e3.mapping == lab) & (e3.axis == an)].set_index("window").reindex(WL)
        xx = np.arange(3) + (k - 1.5) * 0.08
        ax.vlines(xx, sub["shuffle 5%"], sub["shuffle 95%"], color=c, lw=5, alpha=0.25)
        ok = (sub["p shift"] <= 0.05) & sub["beats shuffle"]
        ax.plot(xx, sub.r, color=c, lw=1.3, label=an)
        ax.scatter(xx[ok.to_numpy()], sub.r[ok], color=c, s=40, zorder=3)
        ax.scatter(xx[~ok.to_numpy()], sub.r[~ok], color="white", edgecolor=c, s=40, zorder=3)
    ax.axhline(0, color="0.7", lw=0.6); ax.axvline(1, color="0.9", lw=10, zorder=0)
    ax.set_xticks(range(3), WL); ax.set_xlabel("pole-analysis window"); ax.set_title(lab, fontsize=10)
axes[0].set_ylabel("Spearman r across windows"); axes[0].legend(fontsize=8, frameon=False, loc="lower left")
fig.suptitle("E3. Pole mappings vs analysis window (filled: beats both the shift null and the block shuffle; "
             "pale bars: block-shuffle 5–95 %)", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "W4_pole_mappings_vs_window"); plt.show()

## E4. Does the tracker find the same motifs with other windows?

Whole tracker re-run with baselines of 2, 10, 20 min (detection 5 min) and detection windows of 3 and 10 min
(baseline 5 min). For each motif of the 5-min library: the best |cos| match in each re-run library, and the
presence by state of the matched motif.

In [9]:
VAR = {"baseline 2 min": "bw120_w300", "baseline 10 min": "bw600_w300", "baseline 20 min": "bw1200_w300",
       "detection 3 min": "bw300_w180", "detection 10 min": "bw300_w600"}
VR = {k: mt.load(REPO / "data" / "processed" / "motif_tracking" / "720762" / "window_sweep" / f"{v}.pkl") for k, v in VAR.items()}
def wstate(c):
    s = sd.state_of_time(np.array([c]), info)[0]
    return s
match = pd.DataFrame(index=[f"M{i + 1}" for i in range(K)])
idx = {}
for k, r in VR.items():
    V = np.array([m.template for m in r.library])
    C = np.abs(U @ V.T)
    match[k] = C.max(1); idx[k] = C.argmax(1)
match.loc["library size"] = [len(r.library) for r in VR.values()]
match.to_csv(OUT / "E4_library_match.csv")
def presence(r, j):
    ws = r.window_stats[r.window_stats.motif == j].copy()
    ws["state"] = ws.centre.map(wstate)
    return ws.groupby("state").present.mean()
prow = []
for i in range(3):
    p0 = presence(res, i)
    for st, v in p0.items():
        prow.append({"motif": f"M{i + 1}", "run": "5 min (original)", "state": st, "presence": v})
    for k, r in VR.items():
        for st, v in presence(r, idx[k][i]).items():
            prow.append({"motif": f"M{i + 1}", "run": k, "state": st, "presence": v, "match |cos|": match.loc[f"M{i + 1}", k]})
pres = pd.DataFrame(prow); pres.to_csv(OUT / "E4_presence_by_state.csv", index=False)
match.round(2)

,baseline 2 min,baseline 10 min,baseline 20 min,detection 3 min,detection 10 min
M1,1.00,1.00,1.00,1.00,1.00
M2,0.95,0.94,0.98,0.99,0.98
M3,0.44,0.91,0.91,0.36,0.94
M4,0.70,0.93,0.98,0.77,0.99
M5,0.48,0.95,0.87,0.52,0.95
M6,0.61,0.90,0.66,0.37,0.96
M7,0.79,0.83,0.79,0.88,0.69
M8,0.57,0.90,0.45,0.19,0.43
M9,0.64,0.74,0.74,0.46,0.86
library size,5.00,10.00,8.00,3.00,9.00


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.1, 1.6]})
ax = axes[0]
M = match.drop("library size").to_numpy(float)
ax.imshow(M, cmap="Greens", vmin=0.5, vmax=1, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if M[i, j] > 0.85 else "k")
ax.set_xticks(range(len(VAR)), [f"{k}\n({int(match.loc['library size', k])} motifs)" for k in VAR], fontsize=8)
ax.set_yticks(range(K), [f"M{i + 1}" + (" (broadband)" if i == 0 else "") for i in range(K)])
ax.set_title("Best |cos| match of each original motif in the re-run library", loc="left", fontsize=10)
ax = axes[1]
runs_ = ["5 min (original)"] + list(VAR)
sts = ["awake", "induction", "isoflurane"]
w = 0.8 / len(runs_)
RC = ["#1D232B", "#9AA5B1", "#6B7682", "#4A525C", "#E0A030", "#2F7F8F"]
for gi, (mo, st) in enumerate([(m_, s_) for m_ in ("M1", "M2") for s_ in sts]):
    for j, rn in enumerate(runs_):
        v = pres[(pres.motif == mo) & (pres.state == st) & (pres.run == rn)].presence
        ax.bar(gi + (j - (len(runs_) - 1) / 2) * w, v.iloc[0] if len(v) else 0, w * 0.92, color=RC[j], label=rn if gi == 0 else None)
ax.set_xticks(range(6), [f"{mo}\n{st}" for mo in ("M1", "M2") for st in sts], fontsize=8)
ax.set_ylabel("share of windows present"); ax.set_ylim(0, 1.05)
ax.legend(fontsize=7.5, frameon=False, ncol=3, loc="upper left")
ax.set_title("Presence by state of M1 (broadband) and M2 (isoflurane motif) in each re-run", loc="left", fontsize=10)
fig.tight_layout(); save(fig, "W5_retracking_library_and_presence"); plt.show()

## E5. Synthetic ground truth: what does the window do to a known signal?

10 bands, a planted motif whose sign switches every 20–60 s, noise, 2 hours. Two versions: no drift, and a slow
drift along the motif (period 25 min, amplitude 1.5 units). For each window:
- **fake baseline movement** (no-drift version): SD of the baseline along the motif;
- **drift recovered** (drift version): r between the baseline and the planted drift;
- **drift leaked into expression**: r between the 5-min mean of the expression and the planted drift.

In [11]:
def synth(seed, drift_amp, n=7200, nb=10):
    g = np.random.default_rng(seed)
    motif = np.array([1, 1, 1, 1, -1, -1, -1, 1, 1, -1], float)[:nb]; u = motif / np.linalg.norm(motif)
    env = np.empty((n, nb)); pos = 0
    while pos < n:
        d = int(g.integers(20, 60)); env[pos:pos + d] = g.choice([-1, 1]) * motif; pos += d
    env += 0.8 * g.standard_normal(env.shape)
    drift = drift_amp * np.sin(2 * np.pi * np.arange(n) / 1500)
    env += drift[:, None] * u[None, :] * np.sqrt(nb)
    return env, u, drift
rows = []
for seed in range(10):
    for amp, lab in ((0.0, "no drift"), (1.5, "drift")):
        env, u, drift = synth(seed, amp)
        Z = (env - env.mean(0)) / env.std(0)
        seg = np.zeros(len(Z), int)
        for bw, wl in zip(BWS, BWLAB):
            B, S = mt.split_baseline(Z, seg, bw)
            b = B @ u; e = ((Z - B) / S) @ u
            slow_e = pd.Series(e).rolling(300, center=True, min_periods=75).mean().to_numpy()
            ok = np.isfinite(slow_e)
            d_z = drift / env.std(0).mean()
            rows.append({"seed": seed, "data": lab, "window": wl,
                         "baseline SD": b.std(), "pole distance (units)": 2 * np.abs(Z @ u).mean(),
                         "drift recovered r": np.corrcoef(b, drift)[0, 1] if amp else np.nan,
                         "drift leaked r": abs(np.corrcoef(slow_e[ok], drift[ok])[0, 1]) if amp else np.nan})
e5 = pd.DataFrame(rows); e5.to_csv(OUT / "E5_synthetic.csv", index=False)
print(e5.groupby(["data", "window"], sort=False)[["baseline SD", "pole distance (units)", "drift recovered r", "drift leaked r"]].mean().round(3).to_string())

                 baseline SD  pole distance (units)  drift recovered r  drift leaked r
data     window                                                                       
no drift 1 min         1.780                  4.917                NaN             NaN
         2 min         1.366                  4.917                NaN             NaN
         5 min         0.889                  4.917                NaN             NaN
         10 min        0.618                  4.917                NaN             NaN
         20 min        0.429                  4.917                NaN             NaN
drift    1 min         2.461                  4.616              0.837           0.143
         2 min         2.292                  4.616              0.893           0.187
         5 min         2.047                  4.616              0.945           0.431
         10 min        1.660                  4.616              0.959           0.714
         20 min        0.880               

In [12]:
g = e5.groupby(["data", "window"], sort=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
x = np.arange(len(BWLAB))
nd = g.get_group if False else None
sub = e5[e5.data == "no drift"].groupby("window", sort=False)
axes[0].errorbar(x, sub["baseline SD"].mean(), yerr=sub["baseline SD"].std(), color="#1D232B", marker="o", capsize=3)
axes[0].set_title("No real drift: how much the baseline\nmoves anyway (fake movement)", fontsize=10, loc="left")
axes[0].set_ylabel("SD of baseline along the motif\n(units of the band z-scores)")
pdist = e5[e5.data == "no drift"]["pole distance (units)"].mean()
axes[0].axhline(pdist, color="#C2504A", ls="--", lw=1); axes[0].text(4, pdist * 0.92, "distance between the poles", ha="right", va="top", fontsize=8, color="#C2504A")
sub = e5[e5.data == "drift"].groupby("window", sort=False)
axes[1].errorbar(x, sub["drift recovered r"].mean(), yerr=sub["drift recovered r"].std(), color="#C2504A", marker="o", capsize=3)
axes[1].set_title("Real drift: how well the baseline\nrecovers it", fontsize=10, loc="left"); axes[1].set_ylabel("r(baseline, planted drift)"); axes[1].set_ylim(0, 1.05)
axes[2].errorbar(x, sub["drift leaked r"].mean(), yerr=sub["drift leaked r"].std(), color="#E0A030", marker="o", capsize=3)
axes[2].set_title("Real drift: how much leaks into\nthe expression", fontsize=10, loc="left"); axes[2].set_ylabel("|r|(slow expression, planted drift)"); axes[2].set_ylim(0, 1.05)
for ax in axes:
    ax.set_xticks(x, BWLAB); ax.axvline(2, color="0.9", lw=10, zorder=0); ax.set_xlabel("sliding-baseline window")
fig.suptitle("E5. Synthetic ground truth (10 simulated sessions; planted motif switching every 20–60 s, drift period 25 min)",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "W6_synthetic_ground_truth"); plt.show()

## Conclusions
(figure numbers refer to `data/out/2026-11-10_slides_figures/`)

- **12, synthetic ground truth:** every sliding window trades two errors.
  - Short windows wander even with no drift: the 5-min baseline moves by about 1/5 of the pole distance.
  - Long windows leak real drift into the expression: 43 % at 5 min, 71 % at 10 min.
  - 5 min recovers a planted drift about as well as anything (r = 0.95).
- **13:** The state associations barely change between 1- and 20-min baselines, so they are not specific to
  5 min.
- **14:** Inside a condition, the real baseline moves more than data with its 60-s blocks shuffled.
  - The extra is clear under isoflurane: M1 1.60 vs 0.97 at 5 min.
  - About 60 % of the apparent movement is the size window noise alone would produce.
- **15:** The test that matters.
  - **Pooled over a state:** most associations are no stronger than the shuffle. They come from differences
    between conditions (stimulation blocks, dose steps), not from tracking within one.
  - **Inside conditions:**
    - awake pupil and running clearly beat the shuffle with short windows (pupil 0.64 vs 0.30 at 1 min);
    - dose inside the stimulation block beats it at 1, 2 and 10 min, but not at 5 min;
    - pupil inside maintenance does not beat it.
- **16:** Pole mappings, by result:
  - **Robust:** "− pole follows pupil under isoflurane" beats the shuffle for every window on M1, M3 and the
    original axis.
  - **Not robust:** "baseline follows dose within maintenance" mostly does not beat it. It is driven by the
    stimulation-block segment, where dose was also raised, so dose and stimulation are confounded.
  - **Expected:** separation vs dose fails the shuffle by design, because it is an awake vs isoflurane
    difference.
- **17:** Re-running the tracker with baselines of 2, 10 and 20 min and detection windows of 3 and 10 min.
  - M1 and M2 come back every time (|cos| ≥ 0.94), with the same state map: M1 in every isoflurane window,
    M2 isoflurane-only.
  - The minor motifs M3–M9 depend on the window: the library has 3–10 motifs.


### Bottom line

- **Not created by the window:** the main motifs (M1, M2), the state map, the opponency results, the − pole /
  pupil link, and the awake/isoflurane difference.
- **Partly created by window noise or condition differences:**
  - about 60 % of the size of within-condition baseline movement;
  - pooled baseline–state correlations;
  - "baseline follows dose", which is confounded with the stimulation block.
- **Depends on the window:** the minor motifs M3–M9.
- **A better default:** a 2-min baseline lowers chance correlations and keeps the real within-condition links.
  But it wanders more (figure 12), so it suits short-timescale questions.
